In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. ĐỌC DỮ LIỆU
# ==========================================

DATA_DIR = Path("../../dataset")
INPUT_FILE = DATA_DIR / "hasaki_products.csv"
OUTPUT_DIR = DATA_DIR / "cleaned"
OUTPUT_FILE = OUTPUT_DIR / "hasaki_cleaned.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)

print("Kích thước dữ liệu ban đầu:", df.shape)

# 2. ĐỌC DỮ LIỆU
# ==================================================

df = pd.read_csv(INPUT_FILE)

print("\n===== DỮ LIỆU BAN ĐẦU =====")
print("Số dòng:", len(df))
print("Số cột:", len(df.columns))

# ==================================================
# 3. CHUẨN HÓA TÊN CỘT
# ==================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# ==================================================
# 4. LÀM SẠCH DỮ LIỆU VĂN BẢN
# ==================================================

text_cols = df.select_dtypes(include=["object"]).columns

for col in text_cols:
    df[col] = df[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Chuyển chuỗi rỗng thành giá trị thiếu
df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

# ==================================================
# 5. LOẠI BỎ CÁC CỘT KHÔNG CẦN THIẾT
# ==================================================

drop_cols = [
    "sku",
    "order_count_30",
    "stock_quantity",
    "has_deal",
    "has_gift",
    "english_name"
]

existing_drop_cols = [
    col for col in drop_cols if col in df.columns
]

df.drop(columns=existing_drop_cols, inplace=True)

print("\nCác cột đã loại bỏ:", existing_drop_cols)

# ==================================================
# 6. CHUẨN HÓA KIỂU DỮ LIỆU SỐ
# ==================================================

numeric_cols = [
    "product_id",
    "price",
    "original_price",
    "discount_rate",
    "rating_average",
    "review_count",
    "quantity_sold",
    "category_id"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ==================================================
# 7. KIỂM TRA VÀ LOẠI BỎ DÒNG TRÙNG
# ==================================================

rows_before = len(df)

# Loại dòng trùng hoàn toàn
df.drop_duplicates(inplace=True)

# Mỗi product_id chỉ giữ một dòng
# Nếu trùng mã, giữ bản ghi xuất hiện đầu tiên
df.drop_duplicates(
    subset=["product_id"],
    keep="first",
    inplace=True
)

print("\nSố dòng trùng đã loại:", rows_before - len(df))

# ==================================================
# 8. KIỂM TRA GIÁ VÀ ĐIỂM ĐÁNH GIÁ
# ==================================================

invalid_price = (
    df["price"].notna()
    & (df["price"] <= 0)
)

invalid_rating = (
    df["rating_average"].notna()
    & ~df["rating_average"].between(0, 5)
)

print("Số dòng có giá bán <= 0:", invalid_price.sum())
print("Số dòng có điểm đánh giá ngoài khoảng 0-5:",
      invalid_rating.sum())

# Chưa tự động xóa các dòng bất thường
# để tránh loại nhầm dữ liệu hợp lệ.

# ==================================================
# 9. TẠO CỘT SỐ TIỀN GIẢM GIÁ
# ==================================================

# Chỉ tính khi có cả giá gốc và giá bán
df["discount"] = (
    df["original_price"] - df["price"]
).where(
    df["original_price"].notna()
    & df["price"].notna()
)

# ==================================================
# 11. CHUẨN HÓA NGUỒN DỮ LIỆU
# ==================================================

df["source"] = "hasaki"

# ==================================================
# 12. ĐỔI TÊN CỘT SANG TIẾNG VIỆT KHÔNG DẤU
# ==================================================

rename_cols = {
    "source": "Nguồn dữ liệu",
    "product_id": "Mã sản phẩm",
    "name": "Tên sản phẩm",
    "brand": "Thương hiệu",
    "seller_id": "Mã người bán",
    "price": "Giá bán",
    "original_price": "Giá gốc",
    "discount": "Số tiền giảm giá",
    "discount_rate": "Tỷ lệ giảm giá",
    "rating_average": "Điểm đánh giá trung bình",
    "review_count": "Số lượt đánh giá",
    "quantity_sold": "Số lượng đã bán",
    "is_authentic": "Xác thực chính hãng",
    "category_id": "Mã danh mục",
    "category_name": "Tên danh mục",
    "category_path": "Đường dẫn danh mục",
    "url": "Đường dẫn sản phẩm",
    "thumbnail_url": "Đường dẫn ảnh sản phẩm",
    "crawled_at": "Thời gian thu thập"
}

df.rename(columns=rename_cols, inplace=True)

# ==================================================
# 13. ĐỒNG NHẤT 19 CỘT VỚI TIKI
# ==================================================

standard_cols = [
    "Nguồn dữ liệu",
    "Mã sản phẩm",
    "Tên sản phẩm",
    "Thương hiệu",
    "Mã người bán",
    "Giá bán",
    "Giá gốc",
    "Số tiền giảm giá",
    "Tỷ lệ giảm giá",
    "Điểm đánh giá trung bình",
    "Số lượt đánh giá",
    "Số lượng đã bán",
    "Xác thực chính hãng",
    "Mã danh mục",
    "Tên danh mục",
    "Đường dẫn danh mục",
    "Đường dẫn sản phẩm",
    "Đường dẫn ảnh sản phẩm",
    "Thời gian thu thập"
]

# Bổ sung cột còn thiếu để bảo đảm đúng cấu trúc
for col in standard_cols:
    if col not in df.columns:
        df[col] = np.nan

# Sắp xếp đúng thứ tự
df = df[standard_cols]

# ==================================================
# Chuẩn hóa tên danh mục sang tiếng Việt
category_mapping = {
    "cham-soc-da-mat-c4": "Chăm sóc da mặt",
    "cham-soc-co-the-c12": "Chăm sóc cơ thể"
}
df["Tên danh mục"] = (
    df["Tên danh mục"]
    .astype("string")
    .str.strip()
    .replace(category_mapping)
)
# Kiểm tra kết quả
print(df["Tên danh mục"].value_counts(dropna=False))

# ==================================================
# 14. KIỂM TRA GIÁ TRỊ THIẾU
# ==================================================

print("\n===== GIÁ TRỊ THIẾU =====")

missing = df.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

# ==================================================
# 15. LƯU FILE ĐÃ LÀM SẠCH
# ==================================================

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ==================================================
# 16. KIỂM TRA KẾT QUẢ CUỐI CÙNG
# ==================================================

print("\n===== KẾT QUẢ SAU LÀM SẠCH =====")
print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
print("Danh sách cột:", df.columns.tolist())
print("Số mã sản phẩm trùng:",
      df["Mã sản phẩm"].duplicated().sum())
print("File đã lưu tại:", OUTPUT_FILE.resolve())

# Đọc lại file để xác nhận đã xuất thành công
df_check = pd.read_csv(OUTPUT_FILE)

print("\nKiểm tra file đã lưu:")
print("Kích thước file:", df_check.shape)
print("Số cột:", len(df_check.columns))


Kích thước dữ liệu ban đầu: (3871, 22)

===== DỮ LIỆU BAN ĐẦU =====
Số dòng: 3871
Số cột: 22

Các cột đã loại bỏ: ['sku', 'order_count_30', 'stock_quantity', 'has_deal', 'has_gift', 'english_name']

Số dòng trùng đã loại: 0
Số dòng có giá bán <= 0: 0
Số dòng có điểm đánh giá ngoài khoảng 0-5: 0
Tên danh mục
Chăm sóc da mặt    3008
Chăm sóc cơ thể     863
Name: count, dtype: Int64

===== GIÁ TRỊ THIẾU =====
Mã người bán           3871
Xác thực chính hãng    3871
Giá gốc                 154
Số tiền giảm giá        154
dtype: int64

===== KẾT QUẢ SAU LÀM SẠCH =====
Số dòng: 3871
Số cột: 19
Danh sách cột: ['Nguồn dữ liệu', 'Mã sản phẩm', 'Tên sản phẩm', 'Thương hiệu', 'Mã người bán', 'Giá bán', 'Giá gốc', 'Số tiền giảm giá', 'Tỷ lệ giảm giá', 'Điểm đánh giá trung bình', 'Số lượt đánh giá', 'Số lượng đã bán', 'Xác thực chính hãng', 'Mã danh mục', 'Tên danh mục', 'Đường dẫn danh mục', 'Đường dẫn sản phẩm', 'Đường dẫn ảnh sản phẩm', 'Thời gian thu thập']
Số mã sản phẩm trùng: 0
File đã lưu tạ

C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_23700\2645459149.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=["object"]).columns


In [10]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../dataset/cleaned")
INPUT_FILE = DATA_DIR / "hasaki_cleaned.csv"
OUTPUT_FILE = DATA_DIR / "hasaki_cleaned.csv"

df = pd.read_csv(INPUT_FILE)

# Chuyển các cột giá về dạng số
for col in ["Giá bán", "Giá gốc", "Số tiền giảm giá"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Chỉ tính số tiền giảm giá khi có đủ hai mức giá
valid_discount = (
    df["Giá gốc"].notna()
    & df["Giá bán"].notna()
    & (df["Giá gốc"] >= df["Giá bán"])
)

# Điền các ô giảm giá còn thiếu nếu đủ căn cứ
missing_discount = df["Số tiền giảm giá"].isna()

df.loc[
    missing_discount & valid_discount,
    "Số tiền giảm giá"
] = (
    df.loc[missing_discount & valid_discount, "Giá gốc"]
    - df.loc[missing_discount & valid_discount, "Giá bán"]
)

# Kiểm tra số lượng giá trị thiếu
print("Số ô thiếu giá gốc:", df["Giá gốc"].isna().sum())
print("Số ô thiếu số tiền giảm giá:", df["Số tiền giảm giá"].isna().sum())

# Lưu lại file
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Đã lưu dữ liệu tại:", OUTPUT_FILE.resolve())


Số ô thiếu giá gốc: 154
Số ô thiếu số tiền giảm giá: 154


PermissionError: [Errno 13] Permission denied: '..\\..\\dataset\\cleaned\\hasaki_cleaned.csv'